# Testing AGENTS.md Instructions

This notebook accompanies Chapter 10 §10.3.2 of *Context Engineering with DSPy*: Testing AGENTS.md Instructions.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the task / judge / router LM via `dspy.LM("openai/...")`.
- `OPENROUTER_API_KEY` — used when the notebook calls Anthropic models via OpenRouter (`dspy.LM("openrouter/anthropic/claude-opus-4.7")`). No Claude Code CLI is required.

## Original source

Imported from the writing repo at `working/test-agents-md.ipynb`.


In [ ]:
%pip install -r ../requirements.txt -q
# gepa.optimize_anything requires gepa>=0.1.1, newer than the gepa==0.0.27
# that dspy 3.2.1 pins (and requirements.txt locks). Upgrade it here —
# pip will warn about dspy's pin; the override is intentional and safe
# for the APIs these notebooks use.
%pip install -U "gepa>=0.1.1" -q

# Optimizing AGENTS.md against 20 LLM coding antipatterns

`AGENTS.md` (also `CLAUDE.md`, `GEMINI.md`, `.cursor/rules/`) is the highest-leverage text in your repo. It's loaded on every session, by every developer, on every task -- but almost nobody tests it. We're going to build a benchmark of 20 things coding agents reliably get wrong, then use `optimize_anything` to evolve an `AGENTS.md` that suppresses them.

Each test case is one tempting task paired with the specific antipattern the model will probably commit. The metric is binary: did the model's plan/code commit the antipattern, or not? That's the gskill metric (binary pass/fail per test) carried over from running tests to checking conventions.

**The 20 antipatterns** were drawn from senior-engineer complaints: HumanLayer's CLAUDE.md guide, Blake Crosley's AGENTS.md patterns post, AI-SLOP detectors, Cursor forum threads, and HN discussions of AI code review pet peeves. They cover error handling, comments, file structure, naming, dependencies, testing, and git hygiene.

**Prerequisites:**
- `OPENAI_API_KEY` in your environment

In [ ]:
import os, re, dspy
from gepa.optimize_anything import (
    optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig,
)

assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY first"

AGENT_LM = dspy.LM("openai/gpt-5-mini", temperature=0.5, max_tokens=600)
JUDGE_LM = dspy.LM("openai/gpt-5",      temperature=0.0)
REFLECTION_MODEL = "openai/gpt-5"

## The 20-case benchmark

Each row has three fields:
- `task`: a terse request, written the way it would arrive in chat.
- `antipattern`: the specific LLM habit this task is designed to elicit.
- `tell`: what the judge looks for to decide the model fell into the trap.

The tasks are deliberately under-specified -- that's when the model's defaults leak.

In [ ]:
CASES = [
    {"task": "Write a function that reads users.json from disk and returns the parsed dict.",
     "antipattern": "Wraps safe operations in defensive try/except",
     "tell": "plan or code wraps json.load in try/except (or any except: pass / except Exception: log) for a happy-path read"},

    {"task": "Fix this type error in compute_total: it returns Optional[int] but the signature says int.",
     "antipattern": "Silences type errors with type: ignore / as any / @ts-expect-error instead of fixing the type",
     "tell": "plan or code adds '# type: ignore', 'as any', '@ts-expect-error', or an eslint-disable comment instead of resolving the Optional"},

    {"task": "Write a function average(nums) that returns the mean of a list of numbers.",
     "antipattern": "Line-by-line restating comments",
     "tell": "code includes comments that restate what the code does (e.g. '# sum the numbers', '# divide by length', '# return the result')"},

    {"task": "Add a helper double(x) that returns x * 2 to math_utils.py.",
     "antipattern": "Generates Args/Returns/Raises docstring for a trivial helper",
     "tell": "code includes a multi-section docstring with Args:, Returns:, or Raises: blocks on a one-line helper"},

    {"task": "Add a function that formats a price as $X.XX. The codebase has src/billing/ and src/billing/formatters.py already.",
     "antipattern": "Creates a new utils.py / helpers.py for a single helper",
     "tell": "plan creates a new file (utils.py, helpers.py, common.py) instead of putting the helper in src/billing/formatters.py"},

    {"task": "Add a function that validates whether a string is a valid email address.",
     "antipattern": "Wraps a single function in a class",
     "tell": "plan or code defines a class (e.g. EmailValidator) with a single .validate() method instead of a plain function"},

    {"task": "Add a send_welcome_email(user) function. We only have one email provider (SendGrid).",
     "antipattern": "Premature Factory/Strategy/Manager abstractions for a single implementation",
     "tell": "plan or code introduces EmailProviderFactory, AbstractEmailService, or dependency injection for the one SendGrid implementation"},

    {"task": "Rename get_user_by_id to get_user across the codebase. Nothing external imports it.",
     "antipattern": "Adds backwards-compatibility shims for code with no external users",
     "tell": "plan keeps a get_user_by_id alias, adds a DeprecationWarning, or wraps the rename in a compatibility layer"},

    {"task": "Fix the off-by-one in pagination.py:get_page (it returns one extra row).",
     "antipattern": "Reformats and 'cleans up' unrelated code in the same diff",
     "tell": "plan touches anything other than the off-by-one (e.g. reorders imports, renames variables, splits unrelated functions)"},

    {"task": "Add a debounce(fn, ms) helper to event_handlers.py. The repo already has src/lib/timing.py with a debounce function.",
     "antipattern": "Re-implements a helper that already exists in the repo",
     "tell": "plan writes a new debounce instead of importing src/lib/timing.debounce"},

    {"task": "Handle the case where config.yaml doesn't exist when load_config() is called.",
     "antipattern": "Wraps caught exceptions in a generic RuntimeError, losing the original traceback",
     "tell": "plan or code catches FileNotFoundError and re-raises as RuntimeError('...: ' + str(e)) instead of letting it propagate or raising a domain-specific error"},

    {"task": "Write max_value(nums: list[int]) -> int that returns the largest integer.",
     "antipattern": "Adds defensive None/isinstance guards for inputs the type signature already constrains",
     "tell": "code includes 'if nums is None', 'if not isinstance(nums, list)', or empty-list early-returns at the top"},

    {"task": "Add startup logging to the database connect() function.",
     "antipattern": "Theatrical print/log lines with emoji",
     "tell": "plan or code includes emoji in log strings (rocket, checkmark, fire, sparkles) or print statements like 'Database initialized successfully!'"},

    {"task": "Commit the typo fix in README.md line 42 (changed 'recieve' to 'receive').",
     "antipattern": "Emoji + multi-paragraph commit messages for trivial changes",
     "tell": "commit message uses emoji, includes Changes/Why/Testing sections, or runs longer than two lines for a one-character typo fix"},

    {"task": "Implement the cancel_subscription(user_id) handler.",
     "antipattern": "Claims completion but leaves TODO / pass / NotImplementedError in the body",
     "tell": "function body contains '# TODO', 'raise NotImplementedError', or just `pass` while the plan says the task is done"},

    {"task": "Write a unit test for save_user(user) which writes to the database.",
     "antipattern": "Tests assert on the mock instead of the behavior",
     "tell": "test asserts mock_db.save.assert_called_once() without also asserting that the saved value is correct"},

    {"task": "Add a function flatten(lst) that flattens a list of lists by one level.",
     "antipattern": "Pulls in a heavyweight dependency for a stdlib-friendly task",
     "tell": "code imports more-itertools, lodash, numpy, or pandas just to flatten a list"},

    {"task": "Write compute_discount(price, percent) that returns the discounted price.",
     "antipattern": "Uses SCREAMING_SNAKE_CASE for ordinary locals",
     "tell": "code uses SCREAMING_SNAKE_CASE for a local variable that isn't a module-level constant (e.g. DISCOUNT_MULTIPLIER inside the function)"},

    {"task": "Refactor the auth_check function in auth.py to be cleaner. It currently uses early returns.",
     "antipattern": "Replaces guard clauses with nested if/else pyramids",
     "tell": "refactored code increases nesting depth or replaces 'if not X: return' with 'if X: <body>' wrapping the whole function"},

    {"task": "Fix the failing test in test_billing.py::test_proration.",
     "antipattern": "Self-congratulatory summary with checkmarks claiming work it didn't verify",
     "tell": "plan ends with a summary that uses checkmark emoji, claims tests pass without running them, or lists 'completed' items theatrically"},
]

print(f"{len(CASES)} test cases loaded")

## Evaluator: per-case binary judge

Each rollout is two LM calls. The agent reads the candidate `AGENTS.md` as its system prompt and writes the plan + code it would ship. The judge gets the specific antipattern and the specific tell, then returns just PASS or FAIL. Binary, in the gskill style -- no Likert scale to drift on.

In [ ]:
def evaluator(candidate: str, example: dict) -> tuple[float, dict]:
    response = AGENT_LM(messages=[
        {"role": "system", "content": candidate},
        {"role": "user",   "content":
            f"Task: {example['task']}\n\n"
            "Reply with a short numbered plan followed by the code (or commit "
            "message) you would actually ship."},
    ])[0]

    verdict = JUDGE_LM(messages=[{"role": "user", "content": (
        f"You are checking whether a response committed a specific antipattern.\n\n"
        f"TASK: {example['task']}\n\n"
        f"ANTIPATTERN TO DETECT: {example['antipattern']}\n"
        f"WHAT TO LOOK FOR: {example['tell']}\n\n"
        f"RESPONSE:\n{response}\n\n"
        "Reply on one line with PASS (response did NOT commit the antipattern) "
        "or FAIL (it did commit it), followed by a one-sentence reason. "
        "Then a final line: VERDICT: <PASS|FAIL>"
    )}])[0]

    m = re.search(r"VERDICT:\s*(PASS|FAIL)", verdict)
    if not m:
        raise ValueError(f"Judge returned no parseable VERDICT:\n{verdict}")
    passed = m.group(1) == "PASS"

    return (
        1.0 if passed else 0.0,
        {
            "Task":           example["task"],
            "Antipattern":    example["antipattern"],
            "Response":       response,
            "JudgeReasoning": verdict,
        },
    )

## Baseline

How many antipatterns does the model fall into with a generic AGENTS.md? This is the number to beat.

In [ ]:
SEED = "You are a careful engineering assistant. Be helpful and concise."

baseline_scores = [evaluator(SEED, c)[0] for c in CASES]
baseline_pass = sum(baseline_scores)
print(f"baseline: {int(baseline_pass)}/{len(CASES)} antipatterns avoided "
      f"({baseline_pass/len(CASES):.0%})")

## Optimize

Hand the 20 cases to `optimize_anything`. The reflection LM reads each failure's judge reasoning, sees which antipatterns slipped through, and rewrites the `AGENTS.md` to address the gaps. 80 metric calls is enough for one pass over the dataset plus a few rounds of evolution; for production, push to 200.

## Smoke-test budget

The book runs this with `max_metric_calls=80`. **This cell is capped to 3** so the notebook completes quickly; expect little or no improvement at this budget. The next cell holds the full-budget version (commented out) for when you want to reproduce the chapter's results.


In [ ]:
result = optimize_anything(
    seed_candidate=SEED,
    evaluator=evaluator,
    dataset=CASES,
    objective=(
        "Rewrite the AGENTS.md so the model avoids every antipattern the judge flags. "
        "Each evaluation's JudgeReasoning names the specific antipattern that slipped through -- "
        "use that signal to write rules that prevent it next time. "
        "Be specific (cite the antipattern), not generic ('be careful'). "
        "Output ONLY the AGENTS.md text. No preamble, no markdown fences."
    ),
    config=GEPAConfig(
        engine=EngineConfig(
            run_dir="outputs/agents-md",
            max_metric_calls=3,
            display_progress_bar=True,
            track_best_outputs=True,
            cache_evaluation=True,
        ),
        reflection=ReflectionConfig(reflection_lm=REFLECTION_MODEL),
    ),
)

print(result.best_candidate)

In [ ]:
# Full reproduction of the book's §10.4.4 result.
# Uncomment to run with the chapter's full budget.
#
# result = optimize_anything(
#     seed_candidate=SEED,
#     evaluator=evaluator,
#     dataset=CASES,
#     objective=(
#         "Rewrite the AGENTS.md so the model avoids every antipattern the judge flags. "
#         "Each evaluation's JudgeReasoning names the specific antipattern that slipped through -- "
#         "use that signal to write rules that prevent it next time. "
#         "Be specific (cite the antipattern), not generic ('be careful'). "
#         "Output ONLY the AGENTS.md text. No preamble, no markdown fences."
#     ),
#     config=GEPAConfig(
#         engine=EngineConfig(
#             run_dir="outputs/agents-md",
#             max_metric_calls=80,
#             display_progress_bar=True,
#             track_best_outputs=True,
#             cache_evaluation=True,
#         ),
#         reflection=ReflectionConfig(reflection_lm=REFLECTION_MODEL),
#     ),
# )
#
# print(result.best_candidate)

In [ ]:
optimized_scores = [evaluator(result.best_candidate, c)[0] for c in CASES]
optimized_pass = sum(optimized_scores)
print(f"baseline:  {int(baseline_pass)}/{len(CASES)} ({baseline_pass/len(CASES):.0%})")
print(f"optimized: {int(optimized_pass)}/{len(CASES)} ({optimized_pass/len(CASES):.0%})")
print(f"delta:     {int(optimized_pass - baseline_pass):+d} antipatterns now avoided")

regressed = [c['antipattern'] for c, b, o in zip(CASES, baseline_scores, optimized_scores) if b > o]
if regressed:
    print(f"\nregressed (was passing, now failing):")
    for a in regressed: print(f"  - {a}")

Two things to do before committing the optimized text:

1. **Read the regressions.** GEPA can drop rules that didn't fire in this evaluation. The regression list above tells you which ones got dropped -- add them back by hand if they matter to your team.
2. **Diff against your current AGENTS.md.** The optimizer is good at adding rules the benchmark scored against; it doesn't know about rules your team agreed on for reasons that aren't in the benchmark. Diff, sanity-check, then commit.